# Parda — Phase 3: train the PII model (GLiNER) on clean + noisy-OCR text

**Settings:** Accelerator **GPU T4 x1** · Internet **On**
**Inputs (Add Input → Your Work):** your **Parda** (Phase 1) notebook **and** your **Phase 3a** notebook (OCR training data)
**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`

Run cells 1–8 by hand (≈15 min, includes a 2-minute smoke test). If the smoke test looks right,
**Save Version → Save & Run All** for the real training (≈1.5–3 h GPU).

In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Fonts (Hindi + Kannada, so the tests cover all 3 languages) + Tesseract + GLiNER
!bash scripts/setup_kaggle.sh
!bash scripts/setup_ocr.sh
!pip install -q gliner
!python -c "import gliner, torch; print('gliner', gliner.__version__, '| GPU:', torch.cuda.get_device_name(0))"


In [ ]:
# 3. Find the dataset and the Phase 3a OCR training file (should be 30000 images and 10000 OCR pages)
from parda.ocr.run_ocr import find_data, _n_images
from parda.pii.build_data import find_ocr_train
DATA = find_data()
OCR_TRAIN = find_ocr_train()
print("DATA      =", DATA, "| images:", _n_images(DATA))
print("OCR_TRAIN =", OCR_TRAIN, "| pages:", sum(1 for _ in open(OCR_TRAIN)) if OCR_TRAIN else "NOT FOUND - attach the 3a notebook")
W = "/kaggle/working"
BASE = "urchade/gliner_multi_pii-v1"   # multilingual PII GLiNER (fallback: "urchade/gliner_multi-v2.1")

In [ ]:
# 4. Tests (~3 min): every line ok
!python tests/test_synth.py && python tests/test_ocr.py && python tests/test_pii.py

In [ ]:
# 5. Build training data (~1-2 min): clean text of 10k docs + Tesseract text of 10k docs, benchmark excluded
!python -m parda.pii.build_data --data {DATA} --ocr {OCR_TRAIN} --clean_docs 10000 --out {W}/gliner_data --workers 4

In [ ]:
# 6. Tesseract on the 300 benchmark pages (~3 min, same pages as Phase 2)
!python -m parda.ocr.run_ocr --engine tesseract --data {DATA} --out {W}/bench --per_lang 100 --workers 4

In [ ]:
# 7. BEFORE training: rules-only vs off-the-shelf GLiNER (zero-shot) on the benchmark (~3 min)
!python -m parda.pii.evaluate_pii --data {DATA} --ocr {W}/bench/ocr_tesseract.jsonl --systems rules,base,base+rules --base {BASE} --thresholds 0.3,0.5 --out {W}/pii_eval_before > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{W}/pii_eval_before/report_pii.md").read()))

In [ ]:
# 8. SMOKE TEST (~2 min): 10 training steps. Check: loss is a number (not nan) and the demo line
#    at the end prints some entities. If you see NaN -> add --no_fp16 to cells 8 and 9.
!python -m parda.pii.train_gliner --data_dir {W}/gliner_data --out /tmp/gliner_smoke --base {BASE} --smoke

In [ ]:
# 9. REAL TRAINING. A hard step cap is computed from the smoke test's speed so it can NEVER exceed ~4.5 h,
#    even if this GLiNER version ignores the time guard. Saves to /kaggle/working/gliner/final + private HF repo.
import json, math
from huggingface_hub import HfApi
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
BS, EPOCHS, HOURS = 8, 1, 4.5
n_train = json.load(open(f"{W}/gliner_data/stats.json"))["train_examples"]
sec = json.load(open("/tmp/gliner_smoke/train_summary.json"))["sec_per_step"]   # pessimistic (includes warm-up)
steps_needed = math.ceil(n_train / BS) * EPOCHS
MAX_STEPS = min(steps_needed, int(HOURS * 3600 / sec))
print(f"{n_train} examples -> {steps_needed} steps | smoke speed {sec:.2f} s/step -> est. {steps_needed * sec / 3600:.1f} h"
      f" (upper bound) | max_steps = {MAX_STEPS}")
!python -m parda.pii.train_gliner --data_dir {W}/gliner_data --out {W}/gliner --base {BASE} --bs {BS} --max_steps {MAX_STEPS} --max_hours 5 --hf_repo {HF_USER}/parda-gliner-v1


In [ ]:
# 10. AFTER training: compare everything on the benchmark
!python -m parda.pii.evaluate_pii --data {DATA} --ocr {W}/bench/ocr_tesseract.jsonl --systems rules,base,ft,ft+rules --base {BASE} --ft {W}/gliner/final --thresholds 0.3,0.5 --out {W}/pii_eval_after > /dev/null
display(Markdown(open(f"{W}/pii_eval_after/report_pii.md").read()))

In [ ]:
# 11. Collect results to commit to GitHub (download from the Output tab)
!mkdir -p {W}/phase3_results && cp {W}/pii_eval_after/report_pii.* {W}/gliner/train_summary.json {W}/gliner_data/stats.json {W}/phase3_results/
!ls -la {W}/phase3_results